In [0]:
# =============================================================
# autoloader.py
# Layer     : Bronze
# Purpose   : Ingestion only.
#             - Builds source paths from table config
#             - Builds checkpoint paths
#             - Returns a streaming or batch DataFrame
#             - Nothing else. No writes. No schemas. No Delta.
#
# STRICT RULE: No StructType here. No Delta writes here.
#              If you see MERGE INTO here → wrong file.
# =============================================================

# ── Dependencies ─────────────────────────────────────────────
# abfss() and STORAGE_ACCOUNT injected by set_auth_context
# schema comes from schema_manager.get_schema()

# ── Path builders ─────────────────────────────────────────────────────────────

def build_source_path(table_name: str, table_type: str,
                      load_date: str = None) -> str:
    """
    Build the abfss:// source path for a given table.

    Dimension tables → flat file:
        raw/owners.json

    Fact tables → partitioned folder (Auto Loader reads all partitions
    and uses checkpoint to track what it has already processed):
        raw/device_snapshots/
    """
    if table_type == "dimension":
        return abfss("raw", f"{table_name}.json")
    else:
        # Auto Loader will recursively discover all year=/month=/day= partitions
        return abfss("raw", f"{table_name}/")


def build_checkpoint_path(table_name: str) -> str:
    """
    Checkpoint path in the checkpoints container.
    Auto Loader uses this to track which files have been processed.
    Each table gets its own isolated checkpoint directory.
    """
    return abfss("checkpoints", f"bronze/{table_name}/")


def build_schema_location(table_name: str) -> str:
    """
    Auto Loader schema inference location.
    Stored separately from the checkpoint so schema can be
    inspected and evolved independently.
    """
    return abfss("checkpoints", f"schemas/{table_name}/")


# ── Core ingestion functions ──────────────────────────────────────────────────

def read_stream(table_name: str, table_type: str, schema) -> object:
    """
    Return a STREAMING DataFrame using Auto Loader (cloudFiles).

    Uses the provided schema — no inference at runtime.
    cloudFiles.schemaEvolutionMode = addNewColumns allows
    new source columns to flow through without breaking the pipeline.

    Auto Loader tracks processed files via checkpoint, so re-running
    this will only pick up NEW files — true incremental ingestion.

    Args:
        table_name  : e.g. "device_snapshots"
        table_type  : "dimension" or "fact"
        schema      : StructType from schema_manager.get_schema()

    Returns:
        Streaming DataFrame — no action triggered yet
    """
    source_path     = build_source_path(table_name, table_type)
    schema_location = build_schema_location(table_name)

    print(f"[autoloader] Stream source  : {source_path}")
    print(f"[autoloader] Schema location: {schema_location}")

    return (
        spark.readStream
             .format("cloudFiles")
             .option("cloudFiles.format",              "json")
             .option("cloudFiles.schemaLocation",      schema_location)
             .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
             .option("cloudFiles.inferColumnTypes",    "false")   # use our schema
             .option("multiLine",                      "true")
             .option("recursiveFileLookup",             "true")   # walk year=/month=/day=
             .schema(schema)
             .load(source_path)
    )


def read_batch(table_name: str, table_type: str, schema,
               load_date: str = None) -> object:
    """
    Return a BATCH DataFrame for one-time or full-refresh loads.
    Used for dimension tables and backfills.

    For fact tables with load_date, reads only that day's partition:
        raw/device_snapshots/year=2025/month=01/day=01/

    Args:
        table_name  : e.g. "owners"
        table_type  : "dimension" or "fact"
        schema      : StructType from schema_manager
        load_date   : "yyyy-mm-dd" string — only used for fact tables

    Returns:
        Batch DataFrame
    """
    if table_type == "dimension":
        source_path = build_source_path(table_name, table_type)
    elif load_date:
        # Read a specific day partition
        y, m, d     = load_date.split("-")
        source_path = abfss("raw", f"{table_name}/year={y}/month={m}/day={d}/")
    else:
        source_path = build_source_path(table_name, table_type)

    print(f"[autoloader] Batch source: {source_path}")

    return (
        spark.read
             .option("multiLine",            "true")
             .option("recursiveFileLookup",  "true")
             .schema(schema)
             .json(source_path)
    )

print("[autoloader] Loaded. Ready to serve streams and batch reads.")